# 03 — Setup YouTube Data API v3 dan Validasi Koneksi Awal

## Tujuan Tahap

Notebook ini digunakan untuk memulai proses pengambilan data komentar YouTube pada project:

**“Analisis Sentimen Komentar YouTube terhadap Isu Pelemahan Nilai Rupiah dan Dampaknya terhadap Persepsi Daya Beli Masyarakat Menggunakan Python, Jupyter Notebook, GitHub, dan Streamlit.”**

Pada tahap awal ini, fokus utama adalah:

1. Membaca konfigurasi project dari file `.env` dan `config.yaml`.
2. Memastikan API key YouTube Data API v3 dapat terbaca dari `.env`.
3. Memastikan API key tidak ditulis langsung di notebook.
4. Melakukan validasi koneksi awal ke YouTube Data API v3.
5. Menyiapkan fondasi teknis sebelum proses pencarian video dan crawling komentar dilakukan.

## Prinsip Keamanan Data

API key tidak boleh ditampilkan di output notebook dan tidak boleh diunggah ke GitHub. Dataset raw yang masih memuat informasi author komentar juga tidak boleh dipublikasikan ke GitHub.

In [1]:
# ============================================================
# 1. Import Library Dasar
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import yaml
import pandas as pd

from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

print("Library berhasil di-import.")

Library berhasil di-import.


In [2]:
# ============================================================
# 2. Deteksi Project Root
# ============================================================

def find_project_root(start_path=None, markers=("config.yaml", ".env.example")):
    """
    Mencari root folder project berdasarkan file penanda.
    File penanda yang digunakan:
    - config.yaml
    - .env.example
    """
    current_path = Path(start_path or Path.cwd()).resolve()
    
    for path in [current_path] + list(current_path.parents):
        if all((path / marker).exists() for marker in markers):
            return path
    
    raise FileNotFoundError(
        "Project root tidak ditemukan. Pastikan notebook dijalankan dari dalam folder project "
        "dan file config.yaml serta .env.example tersedia."
    )

PROJECT_ROOT = find_project_root()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Membuat folder data/raw jika belum tersedia
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root berhasil ditemukan:")
print(PROJECT_ROOT)
print("\nFolder data/raw tersedia:", DATA_RAW_DIR.exists())

Project root berhasil ditemukan:
D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA MINING\youtube-rupiah-sentiment-analysis

Folder data/raw tersedia: True


## Validasi File `.env`

API key YouTube Data API v3 dibaca dari file `.env`, bukan ditulis langsung di notebook. Pendekatan ini digunakan untuk menjaga keamanan credential agar tidak ikut masuk ke GitHub.

Variabel yang wajib tersedia di file `.env` adalah:

```text
YOUTUBE_API_KEY=isi_api_key_asli

In [5]:
# ============================================================
# 3. Load API Key dari File .env
# ============================================================

ENV_PATH = PROJECT_ROOT / ".env"

if not ENV_PATH.exists():
    raise FileNotFoundError(
        "File .env tidak ditemukan. Pastikan file .env berada di root folder project."
    )

# Load file .env
load_dotenv(dotenv_path=ENV_PATH)

# Ambil API key dari environment variable
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

# Validasi tanpa menampilkan nilai API key
if not YOUTUBE_API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY tidak ditemukan di file .env. "
        "Pastikan formatnya: YOUTUBE_API_KEY=isi_api_key_asli"
    )

print("File .env berhasil dibaca.")
print("YOUTUBE_API_KEY ditemukan.")
print("API key tidak ditampilkan untuk menjaga keamanan credential.")

File .env berhasil dibaca.
YOUTUBE_API_KEY ditemukan.
API key tidak ditampilkan untuk menjaga keamanan credential.


## Validasi File `config.yaml`

File `config.yaml` digunakan untuk menyimpan parameter crawling, seperti keyword pencarian, jumlah video target, bahasa, region, dan parameter crawling komentar.

Pada tahap ini, file konfigurasi dibaca terlebih dahulu agar proses crawling berikutnya lebih rapi, terstruktur, dan mudah direplikasi.

In [6]:
# ============================================================
# 4. Load Konfigurasi dari config.yaml
# ============================================================

CONFIG_PATH = PROJECT_ROOT / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "File config.yaml tidak ditemukan di root folder project."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file) or {}

print("File config.yaml berhasil dibaca.")
print("Top-level key yang tersedia:", list(config.keys()))

File config.yaml berhasil dibaca.
Top-level key yang tersedia: ['project', 'paths', 'youtube', 'preprocessing', 'modeling']


In [7]:
# ============================================================
# 5. Ambil Parameter Awal dari config.yaml
# ============================================================

youtube_config = config.get("youtube", {})
crawling_config = config.get("crawling", {})

keywords = youtube_config.get("keywords", [
    "rupiah melemah daya beli masyarakat",
    "nilai rupiah melemah",
    "dampak rupiah melemah"
])

region_code = youtube_config.get("region_code", "ID")
relevance_language = youtube_config.get("relevance_language", "id")
order = youtube_config.get("order", "relevance")
search_max_results_per_keyword = int(youtube_config.get("search_max_results_per_keyword", 5))

target_comments_min = int(crawling_config.get("target_comments_min", 6000))
target_comments_max = int(crawling_config.get("target_comments_max", 8000))
max_videos = int(crawling_config.get("max_videos", 20))
comments_per_video = int(crawling_config.get("comments_per_video", 500))
text_format = crawling_config.get("text_format", "plainText")

print("Parameter awal berhasil dimuat.")
print("Jumlah keyword:", len(keywords))
print("Region:", region_code)
print("Bahasa relevansi:", relevance_language)
print("Order pencarian:", order)
print("Target komentar:", target_comments_min, "-", target_comments_max)
print("Maksimal video:", max_videos)
print("Komentar maksimal per video:", comments_per_video)

Parameter awal berhasil dimuat.
Jumlah keyword: 3
Region: ID
Bahasa relevansi: id
Order pencarian: relevance
Target komentar: 6000 - 8000
Maksimal video: 20
Komentar maksimal per video: 500


## Membuat Service Object YouTube Data API v3

Setelah API key berhasil dibaca dari `.env`, langkah berikutnya adalah membuat service object YouTube Data API v3 menggunakan library `google-api-python-client`.

Service object ini akan digunakan pada tahap berikutnya untuk:

1. Mencari video berdasarkan keyword.
2. Mengambil metadata video.
3. Mengambil komentar dari setiap video.
4. Menyimpan hasil crawling ke dataset raw.

## Membuat Service Object YouTube Data API v3

Setelah API key berhasil dibaca dari `.env`, langkah berikutnya adalah membuat service object YouTube Data API v3 menggunakan library `google-api-python-client`.

Service object ini akan digunakan pada tahap berikutnya untuk:

1. Mencari video berdasarkan keyword.
2. Mengambil metadata video.
3. Mengambil komentar dari setiap video.
4. Menyimpan hasil crawling ke dataset raw.

In [8]:
# ============================================================
# 7. Validasi Koneksi Awal ke YouTube Data API v3
# ============================================================

def validate_youtube_api_connection(youtube_service):
    """
    Melakukan validasi koneksi awal ke YouTube Data API v3.
    
    Validasi dilakukan dengan membaca metadata channel publik Google Developers.
    Fungsi ini tidak menampilkan API key.
    """
    try:
        request = youtube_service.channels().list(
            part="snippet",
            id="UC_x5XG1OV2P6uZZ5FSM9Ttw",
            maxResults=1
        )
        
        response = request.execute()
        
        if not response.get("items"):
            return {
                "status": "warning",
                "message": "Request berhasil, tetapi data channel tidak ditemukan."
            }
        
        channel_title = response["items"][0]["snippet"].get("title", "Unknown Channel")
        
        return {
            "status": "success",
            "message": "Koneksi ke YouTube Data API v3 berhasil.",
            "sample_channel_title": channel_title
        }
    
    except HttpError as error:
        error_status = getattr(error.resp, "status", "Unknown")
        
        try:
            error_content = json.loads(error.content.decode("utf-8"))
            error_reason = error_content.get("error", {}).get("errors", [{}])[0].get("reason", "Unknown reason")
            error_message = error_content.get("error", {}).get("message", "Unknown message")
        except Exception:
            error_reason = "Tidak dapat membaca reason error."
            error_message = "Tidak dapat membaca message error."
        
        return {
            "status": "failed",
            "http_status": error_status,
            "reason": error_reason,
            "message": error_message
        }

validation_result = validate_youtube_api_connection(youtube)

print("Status validasi:", validation_result["status"])
print("Pesan:", validation_result["message"])

if validation_result["status"] == "success":
    print("Contoh channel publik terbaca:", validation_result["sample_channel_title"])
else:
    print("HTTP Status:", validation_result.get("http_status"))
    print("Reason:", validation_result.get("reason"))
    print("Catatan: Periksa apakah YouTube Data API v3 sudah enabled, API key benar, dan quota masih tersedia.")

NameError: name 'youtube' is not defined